In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from black_scholes import bs_call, bs_put
from implied_vol import implied_vol

os.makedirs('../figures', exist_ok=True)
sns.set_theme(style='darkgrid')
print('Setup OK')


Setup OK


# Volatilité implicite et Volatility Smile

Ce notebook explore deux aspects :

1. **Validation** du solveur `implied_vol` : round-trip sur plusieurs niveaux de sigma.
2. **Smile de volatilité** : calcul de la vol implicite sur toute une chaîne d'options SPY
   et tracé de la courbe IV en fonction du moneyness.


## 1. Validation — round-trip sur plusieurs sigma

On price chaque option avec Black-Scholes à sigma connu, puis on réinjecte le prix
dans `implied_vol` et on vérifie qu'on retrouve la valeur de départ à 1e-6 près.


In [2]:
S, K, T, r = 100.0, 100.0, 1.0, 0.05
sigmas_test = [0.10, 0.20, 0.40]

rows = []
for sigma_true in sigmas_test:
    for opt in ('call', 'put'):
        price = bs_call(S, K, T, r, sigma_true) if opt == 'call' else bs_put(S, K, T, r, sigma_true)
        sigma_rec = implied_vol(price, S, K, T, r, opt)
        err = abs(sigma_rec - sigma_true)
        rows.append({'sigma_vrai': sigma_true, 'type': opt,
                     'prix_BS': price, 'sigma_implicite': sigma_rec,
                     'ecart': err, 'ok': err < 1e-6})

df_val = pd.DataFrame(rows)
print(df_val.to_string(index=False, float_format=lambda x: f'{x:.8f}'))
print(f"\nTous convergés < 1e-6 : {df_val['ok'].all()}")


 sigma_vrai type     prix_BS  sigma_implicite      ecart   ok
 0.10000000 call  6.80495771       0.10000000 0.00000000 True
 0.10000000  put  1.92790016       0.10000000 0.00000000 True
 0.20000000 call 10.45058357       0.20000000 0.00000000 True
 0.20000000  put  5.57352602       0.20000000 0.00000000 True
 0.40000000 call 18.02295145       0.40000000 0.00000000 True
 0.40000000  put 13.14589390       0.40000000 0.00000000 True

Tous convergés < 1e-6 : True


## 2. Smile de volatilité — données de marché SPY

On récupère la chaîne d'options SPY (premier expiry disponible > 20 jours) via `yfinance`.
Pour chaque strike, on calcule la vol implicite du call ou du put mid-price,
puis on trace IV en fonction du moneyness K/S.


In [3]:
import yfinance as yf
from datetime import datetime, timedelta

TICKER = 'SPY'
USE_SYNTHETIC = False   # sera mis à True si yfinance échoue

try:
    tk = yf.Ticker(TICKER)
    S0 = tk.fast_info['lastPrice']
    if S0 is None or S0 <= 0:
        raise ValueError('Prix spot indisponible')

    # Première maturité > 20 jours
    expirations = tk.options
    today = datetime.today()
    expiry = next(e for e in expirations
                  if (datetime.strptime(e, '%Y-%m-%d') - today).days > 20)
    T_exp = (datetime.strptime(expiry, '%Y-%m-%d') - today).days / 365.0

    chain = tk.option_chain(expiry)
    calls = chain.calls.copy()
    puts  = chain.puts.copy()

    # Mid-price
    calls['mid'] = (calls['bid'] + calls['ask']) / 2
    puts['mid']  = (puts['bid']  + puts['ask'])  / 2

    print(f'Ticker : {TICKER}  |  Spot : {S0:.2f}  |  Expiry : {expiry}  |  T = {T_exp:.4f} an(s)')
    print(f'Calls disponibles : {len(calls)}  |  Puts disponibles : {len(puts)}')

except Exception as exc:
    print(f'[yfinance indisponible] {exc}')
    USE_SYNTHETIC = True


Ticker : SPY  |  Spot : 747.39  |  Expiry : 2026-07-24  |  T = 0.0603 an(s)
Calls disponibles : 169  |  Puts disponibles : 162


In [4]:
if USE_SYNTHETIC:
    print('=== DONNÉES SYNTHÉTIQUES (yfinance inaccessible) ===')
    print('On génère une chaîne fictive avec un skew imposé pour démontrer le concept.\n')

    S0     = 500.0
    T_exp  = 45 / 365
    r_rate = 0.05
    expiry = 'synthétique'

    strikes   = np.arange(440, 561, 5, dtype=float)
    moneyness = strikes / S0
    sigma_true = 0.18 + 0.08 * (1 - moneyness) + 0.04 * (1 - moneyness)**2

    calls = pd.DataFrame({
        'strike': strikes,
        'mid': [bs_call(S0, K, T_exp, r_rate, s) for K, s in zip(strikes, sigma_true)],
    })
    puts = pd.DataFrame({
        'strike': strikes,
        'mid': [bs_put(S0, K, T_exp, r_rate, s) for K, s in zip(strikes, sigma_true)],
    })
    print(f'Strikes générés : {len(strikes)}  |  Vol range : [{sigma_true.min():.2%}, {sigma_true.max():.2%}]')
else:
    r_rate = 0.05

## 3. Calcul de la volatilité implicite par strike

On utilise l'option **hors de la monnaie (OTM)** pour chaque strike — les options OTM
sont plus liquides (bid–ask resserré) et donnent une IV plus fiable :

- **Puts OTM** pour $K/S < 1$ (puts hors de la monnaie)
- **Calls OTM** pour $K/S \geq 1$ (calls hors de la monnaie)

Filtre liquidité : spread $\leq 20\%$ du mid, mid $> 0$, IV $\in [1\%, 200\%]$.

In [5]:
results = []

for df_opt, opt_type in [(puts, 'put'), (calls, 'call')]:
    for _, row in df_opt.iterrows():
        K     = float(row['strike'])
        m     = K / S0
        price = float(row['mid'])

        # Sélection OTM uniquement
        if opt_type == 'put'  and m >= 1.0:
            continue
        if opt_type == 'call' and m <  1.0:
            continue

        if pd.isna(price) or price <= 0:
            continue

        # Filtre liquidité (données réelles seulement)
        if not USE_SYNTHETIC and 'bid' in row and 'ask' in row:
            spread = float(row['ask']) - float(row['bid'])
            if spread > 0.20 * price:
                continue

        iv = implied_vol(price, S0, K, T_exp, r_rate, opt_type)
        if np.isnan(iv) or iv < 0.01 or iv > 2.0:
            continue

        results.append({'strike': K, 'moneyness': m, 'IV': iv,
                        'mid': price, 'type': opt_type})

df_smile = pd.DataFrame(results).sort_values('moneyness')
print(f'Strikes après filtrage : {len(df_smile)}')
print(df_smile[['strike', 'moneyness', 'IV', 'type']].to_string(
    index=False, float_format=lambda x: f'{x:.4f}'))

Strikes après filtrage : 182
  strike  moneyness     IV type
500.0000     0.6690 0.5874  put
505.0000     0.6757 0.5821  put
510.0000     0.6824 0.5687  put
515.0000     0.6891 0.5554  put
520.0000     0.6958 0.5491  put
525.0000     0.7024 0.5359  put
530.0000     0.7091 0.5288  put
535.0000     0.7158 0.5157  put
540.0000     0.7225 0.5027  put
545.0000     0.7292 0.4950  put
550.0000     0.7359 0.4868  put
555.0000     0.7426 0.4739  put
560.0000     0.7493 0.4653  put
565.0000     0.7560 0.4524  put
570.0000     0.7627 0.4435  put
575.0000     0.7693 0.4342  put
580.0000     0.7760 0.4247  put
585.0000     0.7827 0.4150  put
590.0000     0.7894 0.4051  put
595.0000     0.7961 0.3950  put
600.0000     0.8028 0.3847  put
605.0000     0.8095 0.3743  put
610.0000     0.8162 0.3659  put
615.0000     0.8229 0.3551  put
620.0000     0.8296 0.3442  put
625.0000     0.8362 0.3349  put
630.0000     0.8429 0.3253  put
635.0000     0.8496 0.3154  put
640.0000     0.8563 0.3066  put
645.0000   

In [6]:
# Vol ATM = IV du strike le plus proche du spot
atm_row = df_smile.iloc[(df_smile['moneyness'] - 1.0).abs().argmin()]
iv_atm  = atm_row['IV']
print(f'Vol ATM (K={atm_row["strike"]:.1f}, K/S={atm_row["moneyness"]:.4f}) : {iv_atm:.4%}')


Vol ATM (K=747.0, K/S=0.9995) : 13.1469%


## 4. Le Volatility Smile / Skew

### Pourquoi la courbe n'est pas plate ?

Dans le modèle de Black-Scholes, la volatilité `sigma` est **constante** : tous les strikes
doivent avoir la même vol implicite. En pratique, la courbe IV vs strike est **inclinée ou
souriante** (smile), ce qui révèle que le marché attribue des prix différents selon le strike.

### Qu'est-ce qu'un skew ?

Sur les indices actions (SPY, CAC 40…), la courbe est typiquement un **skew négatif** :
- Les **puts OTM** (K < S) ont une vol implicite *plus élevée* que les calls OTM.
- Cela reflète une **prime de risque de krach** : les acheteurs de puts OTM paient cher
  une protection contre des baisses brutales (tail risk). Après le krach de 1987,
  ce phénomène est devenu structurel sur les marchés d'indices.

### En quoi cela contredit Black-Scholes ?

Black-Scholes suppose une distribution log-normale des rendements, symétrique et sans queues
épaisses. Or le marché *pricé* comme si la distribution réelle avait :
1. Une **asymétrie négative** (skewness < 0) : baisses plus probables que la lognormale.
2. Des **queues épaisses** (leptokurtosis) : événements extrêmes sous-estimés par BS.

La courbe IV non plate est la *signature* de ces déviations : c'est l'information que
le marché encode dans les prix d'options que BS ne peut pas capturer avec un seul sigma.


In [7]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(df_smile['moneyness'], df_smile['IV'] * 100,
        'o-', color='steelblue', lw=2, ms=5,
        label='Vol implicite — OTM puts + calls')
ax.axhline(iv_atm * 100, color='tomato', lw=1.5, ls='--',
           label=f'Vol ATM = {iv_atm:.2%}')
ax.axvline(1.0, color='grey', lw=1, ls=':', alpha=0.7)

label_src = 'synthétique' if USE_SYNTHETIC else 'marché'
ax.set_title(f'Volatility Skew — {TICKER}  |  Expiry {expiry}  |  données {label_src}',
             fontsize=13)
ax.set_xlabel('Moneyness  K / S')
ax.set_ylabel('Volatilité implicite (%)')
ax.legend()

# Annotation skew OTM
ax.annotate('Puts OTM : prime de risque\nde krach (skew négatif)',
            xy=(0.93, df_smile[df_smile['moneyness'] < 0.96]['IV'].max() * 100),
            xytext=(0.85, df_smile['IV'].max() * 100 * 0.85),
            arrowprops=dict(arrowstyle='->', color='tomato'),
            fontsize=9, color='tomato')

plt.tight_layout()
out_path = '../figures/07_volatility_smile.png'
plt.savefig(out_path, dpi=150)
plt.show()
print(f'Saved: figures/07_volatility_smile.png')

Saved: figures/07_volatility_smile.png


C:\Users\octav\AppData\Local\Temp\ipykernel_38076\2163277404.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Synthèse


In [8]:
summary = df_smile.copy()
summary['IV (%)'] = (summary['IV'] * 100).round(2)
summary['K/S']    = summary['moneyness'].round(4)
summary['OTM']    = summary['moneyness'].apply(
    lambda m: 'put OTM' if m < 0.98 else ('call OTM' if m > 1.02 else 'ATM'))

print(summary[['strike', 'K/S', 'IV (%)', 'OTM']].to_string(index=False))
print(f'\nVol ATM   : {iv_atm:.4%}')
print(f'Vol min   : {df_smile["IV"].min():.4%}  (K/S = {df_smile.loc[df_smile["IV"].idxmin(), "moneyness"]:.3f})')
print(f'Vol max   : {df_smile["IV"].max():.4%}  (K/S = {df_smile.loc[df_smile["IV"].idxmax(), "moneyness"]:.3f})')
print(f'Amplitude : {(df_smile["IV"].max() - df_smile["IV"].min())*100:.2f} vol points')


 strike    K/S  IV (%)      OTM
  500.0 0.6690   58.74  put OTM
  505.0 0.6757   58.21  put OTM
  510.0 0.6824   56.87  put OTM
  515.0 0.6891   55.54  put OTM
  520.0 0.6958   54.91  put OTM
  525.0 0.7024   53.59  put OTM
  530.0 0.7091   52.88  put OTM
  535.0 0.7158   51.57  put OTM
  540.0 0.7225   50.27  put OTM
  545.0 0.7292   49.50  put OTM
  550.0 0.7359   48.68  put OTM
  555.0 0.7426   47.39  put OTM
  560.0 0.7493   46.53  put OTM
  565.0 0.7560   45.24  put OTM
  570.0 0.7627   44.35  put OTM
  575.0 0.7693   43.42  put OTM
  580.0 0.7760   42.47  put OTM
  585.0 0.7827   41.50  put OTM
  590.0 0.7894   40.51  put OTM
  595.0 0.7961   39.50  put OTM
  600.0 0.8028   38.47  put OTM
  605.0 0.8095   37.43  put OTM
  610.0 0.8162   36.59  put OTM
  615.0 0.8229   35.51  put OTM
  620.0 0.8296   34.42  put OTM
  625.0 0.8362   33.49  put OTM
  630.0 0.8429   32.53  put OTM
  635.0 0.8496   31.54  put OTM
  640.0 0.8563   30.66  put OTM
  645.0 0.8630   29.60  put OTM
  650.0 